In [1]:
# imports
import numpy as np
import pandas as pd
from numpy import random
import scipy
import matplotlib.pyplot as plt
import math
import endecrypt # might need to download this module
import time
import random

# + base is 0 and 90 degrees in the half wave plate
# x base is 45 and -45 degrees in the half wave plate

# See experiment instructions for theory explanation and experiment steps

In [2]:
# SIMULATION WITHOUT EVE
#Functions
def b92_random_without_eve(length):
    alice_bits = [random.randint(0, 1) for _ in range(length)]  # Generate random bits for Alice
    
    # Alice encodes the bits into non-orthogonal states
    alice_states = ['|0>' if bit == 0 else '|+>' for bit in alice_bits]
    
    # Bob randomly chooses bases for measurement
    bob_bases = ['+' if random.random() < 0.5 else 'x' for _ in range(length)]
    
    return alice_bits, alice_states, bob_bases,


def b92_simulation_without_eve(alice_states, bob_bases):
    bob_bits = []
    bob_bit_positions = []
    n=len(alice_states)
   
        
    for j in range(n):
        if bob_bases[j] == '+':# If Bob measures in the standard basis
            #if alice_states[j]=='|0>':
             #   continue
            if alice_states[j] == '|+>':
                 #If Eve sends |+>, Bob measures and records the bit as 1
                bob_measure = random.choice(['|0>','|1>'])
                #if bob_measure == '|0>':
                 #       continue
                if bob_measure == '|1>':
                    bob_bits.append(1)
                    bob_bit_positions.append(j)
                    
             
        else:
            bob_bases[j]=='x'  # If Bob measures in the Hadamard basis
            if alice_states[j]=='|0>': #If Eve sends |0>, Bob measures and gets either |+> or |->
                bob_measure = random.choice(['|+>','|->'])
                #if bob_measure=='|+>':
                 #   continue
                if bob_measure == '|->':
                    bob_bits.append(0)
                    bob_bit_positions.append(j)
            #else:
                #alice_states[j]='|+>' # If Eve sends |+>, Bob discards the result
                #continue
    return bob_bits, bob_bit_positions
    
def convert_to_ascending_numbers(bob_bits):
    return [i for i, _ in enumerate(bob_bits)]

def extract_key(alice_bits, bob_bits, converted_list, bob_bit_positions):
    key = []
    for bob_positionx,ord_list in zip(bob_bit_positions, converted_list):
        if alice_bits[bob_positionx] == bob_bits[ord_list]:
            key.append(bob_bits[ord_list])
    return key

def calculate_qber(alice_bits, bob_bits, bob_bit_positions, converted_list):
    total_bits = len(bob_bit_positions)
    error_bits = 0
    mismatch=[]
    match_bits=0

    for alice_position, bob_position in zip(bob_bit_positions, converted_list):
        if alice_bits[alice_position] == bob_bits[bob_position]:
            continue
        else:    
            error_bits += 1
            mismatch.append([alice_position,bob_position])

    qber = (error_bits / total_bits)*100
    matching_bits = total_bits - error_bits
    if len(mismatch) > 0:
        print('Eavesdropping is detected. Abort the protocol!')

    return qber,mismatch,matching_bits

def key_efficiency(key,alice_bits):
    eff=(len(key)/len(alice_bits))*100
    return eff

def extract_elements(mismatch, index):
    extracted_elements = [inner_list[index] for inner_list in mismatch]
    return extracted_elements

def eliminate_error_bits(bob_bits, extracted_elements):
    # Sort the positions in reverse order to avoid index shifting
    extracted_elements.sort(reverse=True)
    
    # Eliminate elements at the specified positions using list comprehension
    filtered_list = [bob_bits[i] for i in range(len(bob_bits)) if i not in extracted_elements]
    
    return filtered_list

def calculate_asymmetry(key):
    # Count the number of '0's and '1's in the key
    count_0 = key.count(0)
    count_1 = key.count(1)
    
    # Calculate the difference between the counts
    asymmetry = abs(count_0 - count_1)
    
    # Normalize the difference by dividing by the total number of bits
    total_bits = len(key)
    asym = (asymmetry / total_bits)*100
    
    return asym


def shorten_key(data,key):
    # shortens key if it is longer than the given data
    # the key should be the same length as the data for the binary addition
    if len(key) < len(data):
        print('Error - key should be longer than the data')
        return 'Error!'
    return [key[i] for i in range(len(data))]


def encryption(data,key):
    # function which creates the encrypted message using the key created between Alice and Bob
    # using simple binary addition
    # Used by Alice
    
    if len(data) != len(key):
        # if the key is longer than the data, shorten the key
        key = shorten_key(data,key)
    
    encrypted_message = [None for j in range(len(data))]
    
    # binary addition
    for i in range(len(data)):
        if (data[i] == 0 and key[i] == 0) or (data[i] == 1 and key[i] == 1):
            encrypted_message[i] = 0
        elif (data[i] == 1 and key[i] == 0) or (data[i] == 0 and key[i] == 1):
            encrypted_message[i] = 1
       
            
    return encrypted_message

def decryption(message,key):
    # function which recreates the original message using the key created between Alice and Bob
    # The message argument here is the encrypted message Bob recieves from Alice
    # This function is the same as the encryption function, using simple binary addition
    # Used by Bob
    return encryption(message,key)

def string_to_binary(string):
    # finds the binary value of the message
    # Notice for this experiment purposes, the first two integers are discarded for the used message
    # Should generally be 3, but missing the first zero in this case
    # These first integers are used to differ from lower/upper case letters, which is not used in this experiment for the given key
    return endecrypt.encode(string, 'binary')

def binary_to_string(binary):
    # finds the string of the binary value
    # input should be a single string of 1's and 0's accordingly with whitespace between each letter
    # Should enter each letter with the first three integers: Lowercase - 011, Uppercase - 010
    return endecrypt.decode(binary, 'binary')

def string_converter(binary_lst):
    # this function creates an alphabet message from a the binary code
    word_length = 5 # as explained, every word is given as a 5 letter binary code for the message sent
    
    # split the list to different words
    temp = [binary_lst[i:i + 5] for i in range(0, len(binary_lst), 5)]

    bin_data = ''

    for i in range(len(temp)):
        temp[i] = [0,1,0] + temp[i] # add the uppercase digits, can change to lowercase using [0,1,1]
        str_temp = ' ' # notice the whitespace at the beginning of every letter
        res_temp = ' ' # another temporary variable
        for j in range(len(temp[i])): 
            # loop through every word
            str_temp = str(temp[i][j]) # change to string values
            res_temp += str_temp
        bin_data += res_temp
    
    bin_data = bin_data[1:] # delete the first whitespace from the string
    
    binary_values = bin_data.split() # split on whitespace to convert each letter separatley

    res_string = ""
    
    for binary_value in binary_values:
        temp_int = int(binary_value, 2) # create binary value of item
        temp_char = chr(temp_int) # find the letter using the binary alphabet
        res_string += temp_char
    
    return res_string

def binary_converter(string):
    # converts string or message to binary list which corresponds to the message Alice wants to send
    # initialize lists 
    temp_lst = []
    res_lst = []
    
    for character in string:
        # convert to binary
        temp_lst.append(bin(ord(character))[2:].zfill(8))
        
    for i in range(len(temp_lst)):
        # delete three first binary numbers as explained for message
        temp_lst[i] = temp_lst[i][3:]
        
    for j in range(len(temp_lst)):
        # create message as one list of binary numbers to send to Bob via the channel
        for k in range(len(temp_lst[j])):
            res_lst.append(int(temp_lst[j][k]))
    return res_lst

################################################################################################################################
#SIMULATION WITH EVE
def random_with_eve(length):
    alice_bits = [random.randint(0, 1) for _ in range(length)]  # Generate random bits for Alice
    
    # Alice encodes the bits into non-orthogonal states
    alice_states = ['|0>' if bit == 0 else '|+>' for bit in alice_bits]
    
    # Bob randomly chooses bases for measurement
    bob_bases = ['+' if random.random() < 0.5 else 'x' for _ in range(length)]
    
    Eve_bases = ['+' if random.random() < 0.5 else 'x' for _ in range(length)]
    
    return alice_bits, alice_states, bob_bases, Eve_bases



def b92_simulation_with_eve(alice_states, bob_bases, Eve_bases):
    bob_bits = []
    eve_states = []
    
    
    # Eve's interception
    bob_bits = []
    eve_states = []
    bob_bit_position=[]
    n=len(alice_states)    
    
    # Eve's interception
    for i in range(n):
        if Eve_bases[i] == '+':  # If Eve measures in the standard basis
            if alice_states[i] == '|0>':  # If Alice sends |0>, Eve sends randomly |0> or |+> to Bob
                eve_state = random.choice(['|0>', '|+>'])
                eve_states.append(eve_state)
            else:  # If Alice sends |+>, Eve randomly gets |0> or |1>, and sends accordingly
                eve_state = random.choice(['|0>', '|1>'])
                if eve_state=='|0>':
                    eve_state=random.choice(['|0>','|+>'])
                    eve_states.append(eve_state)
                else:
                    eve_states.append('|+>')
        else:  # If Eve measures in the Hadamard basis
            if alice_states[i] == '|0>':  # If Alice sends |0>, Eve randomly gets |+> or |->, and sends accordingly
                eve_state = random.choice(['|+>', '|->'])
                if eve_state=='|+>':
                    eve_state=random.choice(['|0>','|+>'])
                    eve_states.append(eve_state)
                else:
                    eve_states.append('|0>')
                
            else:  # If Alice sends |+>, Eve sends randomly |+> or |0> to Bob
                eve_state = random.choice(['|+>', '|0>'])
                eve_states.append(eve_state)
        
            
    
    # Bob's measurement after Eve's interception
    for j in range(n):
        if bob_bases[j] == '+':  # If Bob measures in the standard basis
            #if eve_states[j] == '|0>':  # If Eve sends |0>, Bob measures and does not record the bit
             #   continue
            if eve_states[j] == '|+>':# If Eve sends |+>, Bob measures and records the bit as 1
                bob_measure=random.choice(['|0>','|1>'])
                #if bob_measure=='|0>':
                 #       continue
                if bob_measure == '|1>':
                    bob_bits.append(1)
                    bob_bit_position.append(j)
                    
             
        else:  # If Bob measures in the Hadamard basis
            if eve_states[j] == '|0>':  # If Eve sends |0>, Bob measures and gets either |+> or |->
                bob_measure = random.choice(['|+>','|->'])
                #if bob_measure=='|+>':
                 #   continue
                if bob_measure == '|->':
                    bob_bits.append(0)
                    bob_bit_position.append(j)
            #else:
             #   eve_states[j] == '|+>'  # If Eve sends |+>, Bob discards the result
              #  continue
    
    return bob_bits, eve_states, bob_bit_position



In [11]:
# Part A - Alice & Bob 50 bit example
# Simulation
print('Theory:')
alice_bit_theory= [1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1]

Alice_states_theory=['|+>', '|+>', '|0>', '|+>', '|0>', '|+>', '|+>', '|0>', '|+>', '|+>', '|+>', '|0>', '|+>', '|0>', '|+>', '|+>', '|0>', '|+>', '|0>', '|0>', '|+>', '|0>', '|+>', '|+>', '|0>', '|0>', '|0>', '|0>', '|0>', '|+>', '|+>', '|+>', '|0>', '|+>', '|0>', '|0>', '|+>', '|+>', '|+>', '|+>', '|0>', '|0>', '|+>', '|+>', '|+>', '|0>', '|+>', '|+>', '|+>', '|+>']

bob_base_theory=['x', '+', 'x', '+', 'x', 'x', 'x', 'x', 'x', 'x', 'x', 'x', '+', 'x', '+', 'x', '+', '+', 'x', 'x', 'x', 'x', 'x', 'x', 'x', '+', 'x', '+', 'x', 'x', '+', 'x', 'x', '+', '+', 'x', '+', '+', '+', 'x', 'x', '+', 'x', 'x', 'x', '+', '+', 'x', '+', '+']

print('\n','Alice bits:',alice_bit_theory)
print('\n','Alice states:',Alice_states_theory)
print('\n','Bob Base:',bob_base_theory)

bob_bit_theory,bob_bit_position_theory = b92_simulation_without_eve(Alice_states_theory,bob_base_theory)
print('\n','Bob bit position:',bob_bit_position_theory)
print('\n','Bob bit theory:',bob_bit_theory)
conv_list_theory=convert_to_ascending_numbers(bob_bit_theory)
key_theory=extract_key(alice_bit_theory, bob_bit_theory, conv_list_theory, bob_bit_position_theory)
print('\n','key_theory:',key_theory)
qber_theory,mismatch_theory,matching_bits_theory=calculate_qber(alice_bit_theory, bob_bit_theory, bob_bit_position_theory, conv_list_theory)
print('\n','matching bits theory:',matching_bits_theory)
print('\n','mis-match bits theory:',len(mismatch_theory))
accuracy_theory=key_efficiency(key_theory,alice_bit_theory)
print('\n','accuracy_theory',accuracy_theory)
###############################################################################################################################
print('Experiment:')
bob_bit_exp=[1,0,0,1,1,1,0,0,0,1,1,1]
bob_bit_position_exp=[1,2,4,12,14,17,19,26,32,36,37,48]
alice_bit_exp=alice_bit_theory
Alice_states_exp=Alice_states_theory
bob_base_exp=bob_base_theory
conv_list_exp=convert_to_ascending_numbers(bob_bit_exp)
key_exp=extract_key(alice_bit_exp, bob_bit_exp, conv_list_exp, bob_bit_position_exp)
print('\n','bob bits exp:',bob_bit_exp)
print('\n','key_exp:',key_exp)
qber_exp,mismatch_exp,matching_bits_exp=calculate_qber(alice_bit_exp, bob_bit_exp, bob_bit_position_exp, conv_list_exp)
print('\n','matching bits experiment:',matching_bits_exp)
print('\n','mis-match bits experiment:',len(mismatch_exp))
accuracy_exp=key_efficiency(key_exp,alice_bit_exp)
print('\n','accuracy_exp',accuracy_exp)


string_to_binary('OK')
binary_to_string('01001111 01001011')

# write message given the binary representation - without leading integers as explained above
message = binary_converter('OK')
print('Alice message:',message)

key=shorten_key(message,key_theory)
print('\n','key:',key)
# Alice creates the encrypted message
encrypted_message = encryption(message,key)
print('\n','Alice sends encrypted message:',encrypted_message)

# Bob recreates the message
decrypted_message = decryption(encrypted_message,key)
print('\n','Bob recreates the message:',decrypted_message)

# after recreating the message
message_bob = string_converter(decrypted_message)
print('\n',"Bob's recreated string message:",message_bob)

Theory:

 Alice bits: [1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1]

 Alice states: ['|+>', '|+>', '|0>', '|+>', '|0>', '|+>', '|+>', '|0>', '|+>', '|+>', '|+>', '|0>', '|+>', '|0>', '|+>', '|+>', '|0>', '|+>', '|0>', '|0>', '|+>', '|0>', '|+>', '|+>', '|0>', '|0>', '|0>', '|0>', '|0>', '|+>', '|+>', '|+>', '|0>', '|+>', '|0>', '|0>', '|+>', '|+>', '|+>', '|+>', '|0>', '|0>', '|+>', '|+>', '|+>', '|0>', '|+>', '|+>', '|+>', '|+>']

 Bob Base: ['x', '+', 'x', '+', 'x', 'x', 'x', 'x', 'x', 'x', 'x', 'x', '+', 'x', '+', 'x', '+', '+', 'x', 'x', 'x', 'x', 'x', 'x', 'x', '+', 'x', '+', 'x', 'x', '+', 'x', 'x', '+', '+', 'x', '+', '+', '+', 'x', 'x', '+', 'x', 'x', 'x', '+', '+', 'x', '+', '+']

 Bob bit position: [2, 4, 7, 12, 13, 19, 24, 26, 36, 37, 40, 46]

 Bob bit theory: [0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1]

 key_theory: [0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1]

 matching bits theory: 1

In [6]:
# Part B - Alice & Bob 90 bit example
alice_bit_theory_90= [0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0]

Alice_states_theory_90 = ['|0>', '|0>', '|+>', '|+>', '|+>', '|+>', '|+>', '|+>', '|+>', '|+>', '|+>', '|+>', '|0>', '|+>', '|0>', '|0>', '|0>', '|0>', '|0>', '|+>', '|+>', '|+>', '|0>', '|0>', '|+>', '|+>', '|+>', '|0>', '|+>', '|0>', '|+>', '|0>', '|+>', '|0>', '|+>', '|+>', '|0>', '|+>', '|+>', '|0>', '|0>', '|0>', '|0>', '|+>', '|0>', '|0>', '|0>', '|0>', '|+>', '|+>', '|+>', '|0>', '|+>', '|0>', '|+>', '|0>', '|0>', '|+>', '|+>', '|+>', '|+>', '|0>', '|+>', '|+>', '|+>', '|0>', '|+>', '|+>', '|0>', '|+>', '|0>', '|+>', '|0>', '|+>', '|+>', '|+>', '|0>', '|0>', '|+>', '|0>', '|0>', '|+>', '|0>', '|0>', '|0>', '|+>', '|+>', '|0>', '|0>', '|0>']

bob_base_theory_90= ['x', 'x', 'x', '+', 'x', 'x', '+', '+', '+', 'x', 'x', '+', 'x', 'x', '+', 'x', 'x', '+', 'x', 'x', '+', '+', 'x', 'x', '+', 'x', 'x', 'x', 'x', 'x', 'x', '+', 'x', 'x', '+', '+', '+', '+', '+', '+', '+', 'x', '+', 'x', '+', 'x', '+', '+', '+', '+', '+', '+', 'x', 'x', '+', '+', '+', '+', '+', '+', '+', '+', 'x', '+', '+', '+', '+', '+', 'x', 'x', '+', '+', '+', 'x', '+', 'x', '+', 'x', 'x', 'x', 'x', 'x', '+', 'x', 'x', 'x', '+', '+', '+', '+']
print('Theory:')

print('\n','Alice bits:',alice_bit_theory_90)
print('\n','Alice states:',Alice_states_theory_90)
print('\n','Bob Base:',bob_base_theory_90)

bob_bit_theory_90,bob_bit_position_theory_90 = b92_simulation_without_eve(Alice_states_theory_90,bob_base_theory_90)
print('\n','Bob bit position:',bob_bit_position_theory_90)
print('\n','Bob bit theory:',bob_bit_theory_90)
conv_list_theory_90=convert_to_ascending_numbers(bob_bit_theory_90)
key_theory_90=extract_key(alice_bit_theory_90, bob_bit_theory_90, conv_list_theory_90, bob_bit_position_theory_90)
print('\n','key_theory:',key_theory_90)
qber_theory_90,mismatch_theory_90,matching_bits_theory_90=calculate_qber(alice_bit_theory_90, bob_bit_theory_90, bob_bit_position_theory_90, conv_list_theory_90)
print('\n','matching bits theory:',matching_bits_theory_90)
print('\n','mis-match bits theory:',len(mismatch_theory_90))
accuracy_theory_90=key_efficiency(key_theory_90,alice_bit_theory_90)
print('\n','accuracy_theory',accuracy_theory_90)
################################################################################################################################
print('\n','Experiment:')

bob_bit_exp_90=[1,1,0,0,0,1,1,0,1,0,1,0,1,1,0,1,1,1,0,1,0,0,1]
bob_bit_position_exp_90=[3,11,12,15,18,20,21,23,24,29,35,45,49,50,53,57,58,67,68,74,79,80,86]
alice_bit_exp_90=alice_bit_theory_90
Alice_states_exp_90=Alice_states_theory_90
bob_base_exp_90=bob_base_theory_90
conv_list_exp_90=convert_to_ascending_numbers(bob_bit_exp_90)
key_exp_90=extract_key(alice_bit_exp_90, bob_bit_exp_90, conv_list_exp_90, bob_bit_position_exp_90)
print('\n','bob bits exp:',bob_bit_exp_90)
print('\n','key_exp:',key_exp_90)
qber_exp_90,mismatch_exp_90,matching_bits_exp_90=calculate_qber(alice_bit_theory_90, bob_bit_exp_90, bob_bit_position_exp_90, conv_list_exp_90)
print('\n','matching bits experiment:',matching_bits_exp_90)
print('\n','mis-match bits experiment:',len(mismatch_exp_90))
accuracy_exp_90=key_efficiency(key_exp_90,alice_bit_exp_90)
print('\n','accuracy_exp',accuracy_exp_90)

string_to_binary('CARL')
binary_to_string('1000011 1000001 1010010 1001100')

# write message given the binary representation - without leading integers as explained above
message_90 = binary_converter('CARL')
print('Alice message:',message_90)

key_90=shorten_key(message_90,key_theory_90)
print('\n','key:',key_90)
# Alice creates the encrypted message
encrypted_message_90 = encryption(message_90,key_90)
print('\n','Alice sends encrypted message:',encrypted_message_90)

# Bob recreates the message
decrypted_message_90 = decryption(encrypted_message_90,key_90)
print('\n','Bob recreates the message:',decrypted_message_90)

# after recreating the message
message_bob_90 = string_converter(decrypted_message_90)
print('\n',"Bob's recreated string message:",message_bob_90)

Theory:

 Alice bits: [0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0]

 Alice states: ['|0>', '|0>', '|+>', '|+>', '|+>', '|+>', '|+>', '|+>', '|+>', '|+>', '|+>', '|+>', '|0>', '|+>', '|0>', '|0>', '|0>', '|0>', '|0>', '|+>', '|+>', '|+>', '|0>', '|0>', '|+>', '|+>', '|+>', '|0>', '|+>', '|0>', '|+>', '|0>', '|+>', '|0>', '|+>', '|+>', '|0>', '|+>', '|+>', '|0>', '|0>', '|0>', '|0>', '|+>', '|0>', '|0>', '|0>', '|0>', '|+>', '|+>', '|+>', '|0>', '|+>', '|0>', '|+>', '|0>', '|0>', '|+>', '|+>', '|+>', '|+>', '|0>', '|+>', '|+>', '|+>', '|0>', '|+>', '|+>', '|0>', '|+>', '|0>', '|+>', '|0>', '|+>', '|+>', '|+>', '|0>', '|0>', '|+>', '|0>', '|0>', '|+>', '|0>', '|0>', '|0>', '|+>', '|+>', '|0>', '|0>', '|0>']

 Bob Base: ['x', 'x', 'x', '+', 'x', 'x', '+', '+', '+', 'x

In [12]:
# Part C - Eve 50 bit example
alice_bit_theory_3= [1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1]

alice_state_theory_3=['|+>', '|+>', '|0>', '|+>', '|0>', '|+>', '|+>', '|0>', '|+>', '|+>', '|+>', '|0>', '|+>', '|0>', '|+>', '|+>', '|0>', '|+>', '|0>', '|0>', '|+>', '|0>', '|+>', '|+>', '|0>', '|0>', '|0>', '|0>', '|0>', '|+>', '|+>', '|+>', '|0>', '|+>', '|0>', '|0>', '|+>', '|+>', '|+>', '|+>', '|0>', '|0>', '|+>', '|+>', '|+>', '|0>', '|+>', '|+>', '|+>', '|+>']

bob_base_theory_3=  ['x', '+', 'x', '+', 'x', 'x', 'x', 'x', 'x', 'x', 'x', 'x', '+', 'x', '+', 'x', '+', '+', 'x', 'x', 'x', 'x', 'x', 'x', 'x', '+', 'x', '+', 'x', 'x', '+', 'x', 'x', '+', '+', 'x', '+', '+', '+', 'x', 'x', '+', 'x', 'x', 'x', '+', '+', 'x', '+', '+']

eve_base_theory=['x', '+', 'x', '+', '+', '+', '+', '+', '+', '+', '+', 'x', 'x', 'x', 'x', 'x', '+', 'x', '+', '+', '+', 'x', 'x', 'x', '+', '+', '+', '+', '+', '+', 'x', 'x', 'x', '+', '+', 'x', '+', 'x', '+', 'x', 'x', '+', '+', '+', 'x', '+', '+', 'x', '+', 'x']

print('\n','Alice Bit theory:',alice_bit_theory_3)
print('\n','Alice states theory:',alice_state_theory_3)
print('\n','Bob Base theory:',bob_base_theory_3)
print('\n','Eve Base theory:',eve_base_theory)

###############################################################################################################################
#Simulation:
print('\nTheory:')
bob_bit_theory_3, eve_state_theory_3, bob_bit_position_theory_3=b92_simulation_with_eve(alice_state_theory_3, bob_base_theory_3, eve_base_theory)
print('\n','bob bits theory:',bob_bit_theory_3)
print('\n','Eve state theory:',eve_state_theory_3)
print('\n','Bob bit position theory:',bob_bit_position_theory_3)


 Alice Bit theory: [1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1]

 Alice states theory: ['|+>', '|+>', '|0>', '|+>', '|0>', '|+>', '|+>', '|0>', '|+>', '|+>', '|+>', '|0>', '|+>', '|0>', '|+>', '|+>', '|0>', '|+>', '|0>', '|0>', '|+>', '|0>', '|+>', '|+>', '|0>', '|0>', '|0>', '|0>', '|0>', '|+>', '|+>', '|+>', '|0>', '|+>', '|0>', '|0>', '|+>', '|+>', '|+>', '|+>', '|0>', '|0>', '|+>', '|+>', '|+>', '|0>', '|+>', '|+>', '|+>', '|+>']

 Bob Base theory: ['x', '+', 'x', '+', 'x', 'x', 'x', 'x', 'x', 'x', 'x', 'x', '+', 'x', '+', 'x', '+', '+', 'x', 'x', 'x', 'x', 'x', 'x', 'x', '+', 'x', '+', 'x', 'x', '+', 'x', 'x', '+', '+', 'x', '+', '+', '+', 'x', 'x', '+', 'x', 'x', 'x', '+', '+', 'x', '+', '+']

 Eve Base theory: ['x', '+', 'x', '+', '+', '+', '+', '+', '+', '+', '+', 'x', 'x', 'x', 'x', 'x', '+', 'x', '+', '+', '+', 'x', 'x', 'x', '+', '+', '+', '+', '+', '+', 'x', 'x', 'x', 

In [13]:
# Part C - 50 bit example - Continue

# Experiment - Same random base and bit as simulation (For comparison)
alice_bit_exp_3=alice_bit_theory_3
alice_state_exp_3=alice_state_theory_3
bob_base_exp_3=bob_base_theory_3
eve_base_exp=eve_base_theory

# From experiment results
bob_bit_exp_3=[0,0,0,1,1,0,0,1,0,0,0,1,1]
bob_bit_position_exp_3=[0,6,8,12,16,18,21,25,29,32,39,45,46]
print('\n','Bob bit experiment:',bob_bit_exp_3)

# Check for Eve
print('Theory:')
conv_list_theory_3=convert_to_ascending_numbers(bob_bit_theory_3)
qber_theory_3,mismatch_theory_3,matching_bits_theory_3=calculate_qber(alice_bit_theory_3, bob_bit_theory_3, bob_bit_position_theory_3, conv_list_theory_3)
key_theory_3=extract_key(alice_bit_exp_3, bob_bit_theory_3, conv_list_theory_3, bob_bit_position_theory_3)
accuracy_theory_3=key_efficiency(key_theory_3,alice_bit_theory_3)
print('Accuracy:',accuracy_theory_3)
print('matching bits:',matching_bits_theory_3)
print('mismatch theory:',len(mismatch_theory_3))

print('\nExperiment:')
conv_list_exp_3=convert_to_ascending_numbers(bob_bit_exp_3)
qber_exp_3,mismatch_exp_3,matching_bits_exp_3=calculate_qber(alice_bit_exp_3, bob_bit_exp_3, bob_bit_position_exp_3, conv_list_exp_3)
key_exp_3=extract_key(alice_bit_exp_3, bob_bit_exp_3, conv_list_exp_3, bob_bit_position_exp_3)
accuracy_exp_3=key_efficiency(key_exp_3,alice_bit_exp_3)
print('Accuracy:',accuracy_exp_3)
print('matching bits:',matching_bits_exp_3)
print('mismatch experiment:',len(mismatch_exp_3))



 Bob bit experiment: [0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1]
Theory:
Eavesdropping is detected. Abort the protocol!
Accuracy: 10.0
matching bits: 5
mismatch theory: 5

Experiment:
Eavesdropping is detected. Abort the protocol!
Accuracy: 10.0
matching bits: 5
mismatch experiment: 8


In [22]:
# Part D - 90 bit example
alice_bit_theory_4= [0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0]

alice_state_theory_4 = ['|0>', '|0>', '|+>', '|+>', '|+>', '|+>', '|+>', '|+>', '|+>', '|+>', '|+>', '|+>', '|0>', '|+>', '|0>', '|0>', '|0>', '|0>', '|0>', '|+>', '|+>', '|+>', '|0>', '|0>', '|+>', '|+>', '|+>', '|0>', '|+>', '|0>', '|+>', '|0>', '|+>', '|0>', '|+>', '|+>', '|0>', '|+>', '|+>', '|0>', '|0>', '|0>', '|0>', '|+>', '|0>', '|0>', '|0>', '|0>', '|+>', '|+>', '|+>', '|0>', '|+>', '|0>', '|+>', '|0>', '|0>', '|+>', '|+>', '|+>', '|+>', '|0>', '|+>', '|+>', '|+>', '|0>', '|+>', '|+>', '|0>', '|+>', '|0>', '|+>', '|0>', '|+>', '|+>', '|+>', '|0>', '|0>', '|+>', '|0>', '|0>', '|+>', '|0>', '|0>', '|0>', '|+>', '|+>', '|0>', '|0>', '|0>']

bob_base_theory_4= ['x', 'x', 'x', '+', 'x', 'x', '+', '+', '+', 'x', 'x', '+', 'x', 'x', '+', 'x', 'x', '+', 'x', 'x', '+', '+', 'x', 'x', '+', 'x', 'x', 'x', 'x', 'x', 'x', '+', 'x', 'x', '+', '+', '+', '+', '+', '+', '+', 'x', '+', 'x', '+', 'x', '+', '+', '+', '+', '+', '+', 'x', 'x', '+', '+', '+', '+', '+', '+', '+', '+', 'x', '+', '+', '+', '+', '+', 'x', 'x', '+', '+', '+', 'x', '+', 'x', '+', 'x', 'x', 'x', 'x', 'x', '+', 'x', 'x', 'x', '+', '+', '+', '+']

eve_base_theory_2= ['+', 'x', '+', 'x', 'x', '+', 'x', '+', 'x', '+', 'x', 'x', '+', '+', 'x', 'x', '+', 'x', '+', 'x', '+', '+', 'x', 'x', 'x', '+', '+', '+', 'x', '+', '+', '+', '+', 'x', '+', '+', 'x', 'x', '+', 'x', '+', 'x', 'x', 'x', '+', 'x', 'x', '+', '+', '+', '+', '+', 'x', 'x', 'x', '+', '+', '+', '+', 'x', '+', '+', 'x', '+', 'x', '+', '+', '+', 'x', 'x', 'x', 'x', '+', 'x', '+', '+', '+', '+', 'x', '+', '+', 'x', '+', '+', 'x', '+', '+', '+', 'x', '+']

print('\n','Alice Bit theory:',alice_bit_theory_4)
print('\n','Alice states theory:',alice_state_theory_4)
print('\n','Bob Base theory:',bob_base_theory_4)
print('\n','Eve Base theory:',eve_base_theory_2)

###############################################################################################################################
#Simulation:
bob_bit_theory_4, eve_state_theory_4, bob_bit_position_theory_4=b92_simulation_with_eve(alice_state_theory_4, bob_base_theory_4, eve_base_theory_2)
print('\n','bob bits theory:',bob_bit_theory_4)
print('\n','Eve state theory:',eve_state_theory_4)
print('\n','Bob bit position theory:',bob_bit_position_theory_4)


 Alice Bit theory: [0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0]

 Alice states theory: ['|0>', '|0>', '|+>', '|+>', '|+>', '|+>', '|+>', '|+>', '|+>', '|+>', '|+>', '|+>', '|0>', '|+>', '|0>', '|0>', '|0>', '|0>', '|0>', '|+>', '|+>', '|+>', '|0>', '|0>', '|+>', '|+>', '|+>', '|0>', '|+>', '|0>', '|+>', '|0>', '|+>', '|0>', '|+>', '|+>', '|0>', '|+>', '|+>', '|0>', '|0>', '|0>', '|0>', '|+>', '|0>', '|0>', '|0>', '|0>', '|+>', '|+>', '|+>', '|0>', '|+>', '|0>', '|+>', '|0>', '|0>', '|+>', '|+>', '|+>', '|+>', '|0>', '|+>', '|+>', '|+>', '|0>', '|+>', '|+>', '|0>', '|+>', '|0>', '|+>', '|0>', '|+>', '|+>', '|+>', '|0>', '|0>', '|+>', '|0>', '|0>', '|+>', '|0>', '|0>', '|0>', '|+>', '|+>', '|0>', '|0>', '|0>']

 Bob Base theory: ['x', 'x', 'x', '+', 'x', 'x', '+', 

In [23]:
# Part C - 90 bit example - Continue

# Experiment - Same random base and bit as simulation (For comparison)
alice_bit_exp_4=alice_bit_theory_4
alice_state_exp_4=alice_state_theory_4
bob_base_exp_4=bob_base_theory_4
eve_base_exp_2=eve_base_theory_2

# From experiment results
bob_bit_exp_4=[0,1,0,1,1,0,1,1,1,1,0,0,1,1,1,1,1,0,1,0,0,0]
bob_bit_position_exp_4=[0,3,5,11,21,23,34,38,39,40,41,43,47,50,51,60,63,69,74,77,79,84]
print('\n','Bob bit experiment:',bob_bit_exp_4)

# Check for Eve
print('Theory:')
conv_list_theory_4=convert_to_ascending_numbers(bob_bit_theory_4)
qber_theory_4,mismatch_theory_4,matching_bits_theory_4=calculate_qber(alice_bit_theory_4, bob_bit_theory_4, bob_bit_position_theory_4, conv_list_theory_4)
key_theory_4=extract_key(alice_bit_theory_4, bob_bit_theory_4, conv_list_theory_4, bob_bit_position_theory_4)
accuracy_theory_4=key_efficiency(key_theory_4,alice_bit_theory_4)
print('Accuracy:',accuracy_theory_4)
print('matching bits:',matching_bits_theory_4)
print('mismatch theory:',len(mismatch_theory_4))

print('\nExperiment:')
conv_list_exp_4=convert_to_ascending_numbers(bob_bit_exp_4)
qber_exp_4,mismatch_exp_4,matching_bits_exp_4=calculate_qber(alice_bit_exp_4, bob_bit_exp_4, bob_bit_position_exp_4, conv_list_exp_4)
key_exp_4=extract_key(alice_bit_exp_4, bob_bit_exp_4, conv_list_exp_4, bob_bit_position_exp_4)
accuracy_exp_4=key_efficiency(key_exp_4,alice_bit_exp_4)
print('Accuracy:',accuracy_exp_4)
print('matching bits:',matching_bits_exp_4)
print('mismatch experiment:',len(mismatch_exp_4))



 Bob bit experiment: [0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0]
Theory:
Eavesdropping is detected. Abort the protocol!
Accuracy: 16.666666666666664
matching bits: 15
mismatch theory: 12

Experiment:
Eavesdropping is detected. Abort the protocol!
Accuracy: 16.666666666666664
matching bits: 15
mismatch experiment: 7


In [43]:
# Part G (i) - 1000 bit example - SIMULATION ONLY

# generate random numbers using the function
random_gen = random_with_eve(1000)

alice_bit_theory = random_gen[0]
alice_state_theory = random_gen[1]
bob_base_theory = random_gen[2]
eve_base_theory = random_gen[3]


# Simulation
# time calculation
start = time.time() # start time
sim_bit = b92_simulation_with_eve(alice_state_theory,bob_base_theory,eve_base_theory)
bob_bit_theory = sim_bit[0]
eve_states_theory = sim_bit[1]
bob_bit_position_theory = sim_bit[2]

# Check for Eve
print('\n','Check For Eve from Simulation:','\n')
conv_list_theory=convert_to_ascending_numbers(bob_bit_theory)
qber,mismatch,matching_bits=calculate_qber(alice_bit_theory, bob_bit_theory, bob_bit_position_theory, conv_list_theory)
key=extract_key(alice_bit_theory, bob_bit_theory, conv_list_theory, bob_bit_position_theory)
accuracy=key_efficiency(key,alice_bit_theory)
print('Accuracy:',accuracy)
print('matching bits:',matching_bits)
print('mismatch experiment:',len(mismatch))

end = time.time() # end time
print('Simulation runtime: ', end - start, ' sec')



 Check For Eve from Simulation: 

Eavesdropping is detected. Abort the protocol!
Accuracy: 15.4
matching bits: 154
mismatch experiment: 85
Simulation runtime:  1.5130114555358887  sec


In [44]:
# Part G (ii) - 10000 bit example - SIMULATION ONLY

# generate random numbers using the function
random_gen = random_with_eve(10000)

alice_bit_theory = random_gen[0]
alice_state_theory = random_gen[1]
bob_base_theory = random_gen[2]
eve_base_theory = random_gen[3]


# Simulation
# time calculation
start = time.time() # start time
sim_bit = b92_simulation_with_eve(alice_state_theory,bob_base_theory,eve_base_theory)
bob_bit_theory = sim_bit[0]
eve_states_theory = sim_bit[1]
bob_bit_position_theory = sim_bit[2]

# Check for Eve
print('\n','Check For Eve from Simulation:','\n')
conv_list_theory=convert_to_ascending_numbers(bob_bit_theory)
qber,mismatch,matching_bits=calculate_qber(alice_bit_theory, bob_bit_theory, bob_bit_position_theory, conv_list_theory)
key=extract_key(alice_bit_theory, bob_bit_theory, conv_list_theory, bob_bit_position_theory)
accuracy=key_efficiency(key,alice_bit_theory)
print('Accuracy:',accuracy)
print('matching bits:',matching_bits)
print('mismatch experiment:',len(mismatch))

end = time.time() # end time
print('Simulation runtime: ', end - start, ' sec')


 Check For Eve from Simulation: 

Eavesdropping is detected. Abort the protocol!
Accuracy: 15.03
matching bits: 1503
mismatch experiment: 940
Simulation runtime:  0.14410638809204102  sec


In [45]:
# Part G (iii) - 1000000 bit example - SIMULATION ONLY

# generate random numbers using the function
random_gen = random_with_eve(1000000)

alice_bit_theory = random_gen[0]
alice_state_theory = random_gen[1]
bob_base_theory = random_gen[2]
eve_base_theory = random_gen[3]


# Simulation
# time calculation
start = time.time() # start time
sim_bit = b92_simulation_with_eve(alice_state_theory,bob_base_theory,eve_base_theory)
bob_bit_theory = sim_bit[0]
eve_states_theory = sim_bit[1]
bob_bit_position_theory = sim_bit[2]

# Check for Eve
print('\n','Check For Eve from Simulation:','\n')
conv_list_theory=convert_to_ascending_numbers(bob_bit_theory)
qber,mismatch,matching_bits=calculate_qber(alice_bit_theory, bob_bit_theory, bob_bit_position_theory, conv_list_theory)
key=extract_key(alice_bit_theory, bob_bit_theory, conv_list_theory, bob_bit_position_theory)
accuracy=key_efficiency(key,alice_bit_theory)
print('Accuracy:',accuracy)
print('matching bits:',matching_bits)
print('mismatch experiment:',len(mismatch))

end = time.time() # end time
print('Simulation runtime: ', end - start, ' sec')


 Check For Eve from Simulation: 

Eavesdropping is detected. Abort the protocol!
Accuracy: 15.6274
matching bits: 156274
mismatch experiment: 93449
Simulation runtime:  6.310755729675293  sec


In [46]:
# Part G (iiii) - 10000000 bit example - SIMULATION ONLY

# generate random numbers using the function
random_gen = random_with_eve(10000000)

alice_bit_theory = random_gen[0]
alice_state_theory = random_gen[1]
bob_base_theory = random_gen[2]
eve_base_theory = random_gen[3]


# Simulation
# time calculation
start = time.time() # start time
sim_bit = b92_simulation_with_eve(alice_state_theory,bob_base_theory,eve_base_theory)
bob_bit_theory = sim_bit[0]
eve_states_theory = sim_bit[1]
bob_bit_position_theory = sim_bit[2]

# Check for Eve
print('\n','Check For Eve from Simulation:','\n')
conv_list_theory=convert_to_ascending_numbers(bob_bit_theory)
qber,mismatch,matching_bits=calculate_qber(alice_bit_theory, bob_bit_theory, bob_bit_position_theory, conv_list_theory)
key=extract_key(alice_bit_theory, bob_bit_theory, conv_list_theory, bob_bit_position_theory)
accuracy=key_efficiency(key,alice_bit_theory)
print('Accuracy:',accuracy)
print('matching bits:',matching_bits)
print('mismatch experiment:',len(mismatch))

end = time.time() # end time
print('Simulation runtime: ', end - start, ' sec')


 Check For Eve from Simulation: 

Eavesdropping is detected. Abort the protocol!
Accuracy: 15.63917
matching bits: 1563917
mismatch experiment: 936435
Simulation runtime:  55.45581412315369  sec
